# Zolai Web Crawler v3 — Bible-Enhanced
**Uses Bible vocabulary to score, filter, and collect Zolai web content**

This notebook:
1. Loads Bible vocabulary patterns from the Bible pipeline output
2. **Auto-discovers Zolai websites** via DuckDuckGo + Bing search
3. Ranks and confirms best resources
4. Crawls confirmed pages for Zolai content
5. Scores each page using Bible word matches
6. Filters and saves only high-quality Zolai text
7. Combines Bible + crawled data into a unified training dataset

**Kaggle Setup:**
- Add Input: `zolai-bible-dataset` (from Bible pipeline)
- Toggle Internet: ON (required for search + crawl)
- No manual URL input needed — auto-discovers resources

In [ ]:
#%% Cell 2: Environment Check & Dependencies
import os, sys, platform, json, hashlib, re, time, socket, unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime
from urllib.parse import urljoin, urlparse

print(f"Platform: {platform.platform()}")
print(f"Python:   {sys.version}")
print(f"CWD:      {os.getcwd()}")

IS_KAGGLE = Path("/kaggle").exists()
WORK_DIR = Path("/kaggle/working" if IS_KAGGLE else ".").resolve()
print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")

# Internet check
def has_internet():
    try:
        socket.create_connection(("8.8.8.8", 53), timeout=3)
        return True
    except OSError:
        return False

INTERNET = has_internet()
print(f"Internet: {'ON' if INTERNET else 'OFF'}")

if INTERNET:
    !pip install -q -U "requests>=2.31.0" "beautifulsoup4>=4.12.0" "httpx>=0.25.0" "tqdm>=4.66" "datasets>=2.19.0" "pandas>=2.0,<3"
    print("Install complete.")
else:
    print("CRITICAL: Internet is OFF. Enable Internet in Kaggle Settings.")

import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

print("\nAll imports OK")

In [ ]:
#%% Cell 3: Load Bible Vocabulary Patterns

KAGGLE_INPUT = Path("/kaggle/input")


def normalize_zolai_token(token):
    token = unicodedata.normalize("NFKC", str(token).lower())
    token = token.replace("’", "'").replace("‘", "'").replace("`", "'")
    token = re.sub(r"^[^a-z']+|[^a-z']+$", "", token)
    token = token.replace("'", "").replace("-", "")
    return token


def find_bible_data():
    candidates = [
        Path("/kaggle/input/datasets/peterpausianlian/bible-datasets/zolai_bible_dataset"),
        Path("/kaggle/input/datasets/peterpausianlian/zolai-bible-dataset"),
        Path("/kaggle/input/zolai-bible-dataset"),
        Path("/kaggle/working/zolai_bible_pipeline_out"),
    ]
    for candidate in candidates:
        if (candidate / "language_patterns.json").exists():
            return candidate
    if KAGGLE_INPUT.exists():
        for item in KAGGLE_INPUT.rglob("language_patterns.json"):
            return item.parent
    return None


def build_zolai_lexicon(bible_data):
    with open(bible_data / "language_patterns.json", "r", encoding="utf-8") as f:
        patterns = json.load(f)

    top_words = []
    top_bigrams = []
    for source_name in ("tdb77", "tedim1932"):
        source_patterns = patterns.get(source_name, {})
        top_words.extend(
            normalize_zolai_token(w) for w, _ in source_patterns.get("top_words", [])
        )
        top_bigrams.extend(
            " ".join(
                normalize_zolai_token(x)
                for x in ng
                if normalize_zolai_token(x)
            )
            for ng, _ in source_patterns.get("top_bigrams", [])
        )
    top_words = [w for w in top_words if w]
    top_bigrams = [bg for bg in top_bigrams if bg.strip()]

    verb_suffixes = {
        "hi", "hiam", "ding", "ciang", "ciangin", "man", "manin",
        "sak", "khia", "lut", "pai", "ciah", "zo", "mu", "ci",
        "piak", "lang", "thiam", "thei", "suak", "sim", "gelh",
        "bawl", "sem", "nei", "zui", "tung", "ngen", "thum",
    }

    verbs = {
        "om", "hi", "ahi", "ci", "kici", "gen", "hilh", "sim", "gelh", "sin",
        "thei", "theih", "tel", "ngai", "ngaihsut", "mu", "muh", "kimu",
        "pai", "paikhia", "hong", "lut", "suan", "tung", "tungsak", "zui",
        "kizui", "nei", "ne", "dawn", "bawl", "sem", "sep", "koi", "koih",
        "lak", "pia", "piak", "ngah", "zang", "khen", "khenkhia", "kikhensak",
        "kikhawm", "piang", "piangsak", "piangkhia", "gamtang", "vaksak", "sak",
        "suaksak", "damsak", "paisak", "kipsak", "omsak", "honkhia", "hawlkhia",
        "thuak", "thum", "bia", "thunget", "du", "duh", "it", "hua", "si",
        "thi", "dam", "nuntak", "zo", "khinzo", "ciah", "bei", "khia", "mai",
        "gawt", "suhkha", "suakta", "huh", "gup", "hehpih", "phat", "zah", "za",
        "hopih", "dong", "dawh", "khawng", "pek", "pe", "la", "thatsak",
        "kilang", "lang", "khenthang", "puak", "puan", "kem", "kep", "kai",
        "khawm", "talsuak", "lutkhia", "man", "mang", "mangngilh", "thupiak",
        "hilhchian", "up", "ngahkhiat", "kipia", "kinep",
    }
    nouns = {
        "pasian", "topa", "jesu", "kha", "lai", "thu", "thupha", "thupiak",
        "leitung", "lei", "vantung", "van", "ni", "zan", "zing", "nitak",
        "khuavak", "khuamial", "tui", "tuipi", "gam", "khua", "khuapi", "inn",
        "mite", "mi", "mihing", "naupang", "pasal", "nupi", "tapa", "tani",
        "sihna", "nuntakna", "gupna", "itna", "upna", "lametna", "pilna",
        "biakna", "biakinn", "pawlpi", "zogam", "zomi", "zolai", "tedim",
    }
    adjectives = {
        "hoih", "sia", "pha", "lian", "neu", "thak", "lui", "hat", "niam",
        "sang", "tawn", "nop", "mahmah", "lua", "siangtho", "dik", "manpha",
    }
    pronouns = {
        "kei", "keimah", "nang", "nangmah", "amah", "amahmah", "ei", "i",
        "eite", "amau", "amaute", "kuamah", "koimah",
    }
    particles = {
        "a", "leh", "in", "na", "uh", "pen", "te", "teng", "pawl", "ciangin",
        "ahih", "ahihmanin", "hangin", "panin", "tawh", "ding", "dingin",
        "hileh", "teh", "zong", "zongin", "bek", "mah", "lo", "hiam",
    }

    stream_word_counts = Counter()
    stream_bigram_counts = Counter()
    bible_stream_file = None
    for candidate in [bible_data / "bible_zolai_combined.jsonl", bible_data / "bible_tdb77.jsonl"]:
        if candidate.exists():
            bible_stream_file = candidate
            break

    rows_streamed = 0
    if bible_stream_file is not None:
        with open(bible_stream_file, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                entry = json.loads(line)
                tokens = [normalize_zolai_token(tok) for tok in re.findall(r"[A-Za-z'\-]+", entry.get("text", ""))]
                tokens = [tok for tok in tokens if len(tok) >= 2 and tok != "note"]
                if not tokens:
                    continue
                stream_word_counts.update(tokens)
                stream_bigram_counts.update(" ".join(pair) for pair in zip(tokens, tokens[1:]))
                rows_streamed += 1

    vocab = set(top_words) | verbs | nouns | adjectives | pronouns | particles
    vocab |= {word for word, count in stream_word_counts.items() if count >= 20 and word != "note"}
    top_bigrams = sorted(set(top_bigrams) | {bg for bg, count in stream_bigram_counts.items() if count >= 25})

    return {
        "patterns": patterns,
        "top_words": top_words,
        "top_bigrams": top_bigrams,
        "verb_suffixes": verb_suffixes,
        "verbs": verbs,
        "nouns": nouns,
        "adjectives": adjectives,
        "pronouns": pronouns,
        "particles": particles,
        "vocab": vocab,
        "rows_streamed": rows_streamed,
    }


BIBLE_DATA = find_bible_data()
if BIBLE_DATA is None:
    raise FileNotFoundError(
        "Bible dataset not found. Add the Zolai Bible dataset as a Kaggle Input."
    )

print(f"Bible data: {BIBLE_DATA}")

LEXICON = build_zolai_lexicon(BIBLE_DATA)
patterns = LEXICON["patterns"]
zolai_top_words = LEXICON["top_words"]
zolai_top_bigrams = LEXICON["top_bigrams"]
VERB_SUFFIXES = LEXICON["verb_suffixes"]
zolai_verbs = LEXICON["verbs"]
zolai_nouns = LEXICON["nouns"]
zolai_adjectives = LEXICON["adjectives"]
zolai_pronouns = LEXICON["pronouns"]
zolai_particles = LEXICON["particles"]
zolai_vocab = LEXICON["vocab"]

print(f"\nLoaded Bible vocabulary:")
print(f"  Streamed rows: {LEXICON['rows_streamed']:,}")
print(f"  Top words: {len(zolai_top_words)}")
print(f"  Top bigrams: {len(zolai_top_bigrams)}")
print(f"  Vocab size: {len(zolai_vocab)}")
print(f"  Verbs: {len(zolai_verbs)}")
print(f"  Sample words: {', '.join(sorted(list(zolai_vocab))[:20])}")


In [ ]:
#%% Path Checker — Show all available inputs
print("=" * 60)
print("KAGGLE INPUT PATH CHECKER")
print("=" * 60)

base = Path("/kaggle/input/datasets/peterpausianlian")
alt_base = Path("/kaggle/input")

print(f"\nBase path: /kaggle/input/datasets/peterpausianlian/")
if base.exists():
    print(f"  EXISTS: {base}")
    for item in sorted(base.iterdir()):
        if item.is_dir():
            subitems = list(item.iterdir())[:5]
            subnames = [p.name for p in subitems]
            more = f" (+{len(list(item.iterdir()))-5} more)" if len(list(item.iterdir())) > 5 else ""
            print(f"  /- {item.name}/")
            for s in subnames:
                print(f"     |- {s}")
            if more:
                print(f"     {more}")
else:
    print(f"  NOT FOUND: {base}")

print(f"\nAlt base: /kaggle/input/")
if alt_base.exists():
    for item in sorted(alt_base.iterdir()):
        if item.is_dir() and item.name not in (".", ".."):
            print(f"  /- {item.name}/")

print(f"\nWorking dir: /kaggle/working/")
working = Path("/kaggle/working")
if working.exists():
    for item in sorted(working.iterdir()):
        print(f"  |- {item.name}")

print("\n" + "=" * 60)

In [ ]:
#%% Cell 4: Link Discovery — Auto-Search for Zolai Resources
# Searches multiple engines to discover Zolai websites, blogs, forums

# Search queries — mix of Zolai, Tedim, Chin keywords
SEARCH_QUERIES = [
    "Zolai language",
    "Zolai Bible",
    "Tedim language",
    "Tedim Chin",
    "Chin language Myanmar",
    "Zolai literature",
    "Tedim dictionary",
    "Chin state language",
    "Zolai news",
    "Tedim culture",
    "Chin Christian",
    "Zolai hymn",
    "Tedim grammar",
    "Chin language learning",
    "Zolai alphabet",
    "Zolai orthography",
    "Tedim Chin Bible translation",
    "Chin Hills language",
    "Zomi language",
    "Zomi dictionary",
    "Zogam culture",
    "Chin state education",
    "Tedim township",
    "Chin literature Myanmar",
    "Zolai writing system",
    "Chin ethnic group",
    "Tedim pronunciation",
    "Chin poetry",
    "Zolai vocabulary",
    "Chin traditional stories",
    "Tedim folk tales",
    "Chin history",
    "Zomi Christian literature",
    "Chin language resources",
    "Tedim Chin dictionary online",
    "Chin language preservation",
    "Zolai text corpus",
    "Chin Bible verses",
    "Tedim language lessons",
    "Chin language translation",
    "Zomi Tedim Chin differences",
    "Chin linguistic study",
    "Zolai Unicode",
    "Chin fonts download",
    "Tedim Chin proverbs",
    "Chin language grammar rules",
    "Zolai keyboard",
    "Chin language Wikipedia",
    "Tedim Chin songs lyrics",
    "Chin state news",
    "Zomi language resources",
    "Chin ethnic languages",
    "Tedim Chin alphabet letters",
    "Chin language documentation",
    "Zolai script history",
    "Chin missionary language work",
    "Tedim Chin vocabulary list",
    "Chin language family Tibeto-Burman",
    "Zomi literature Myanmar",
    "Chin language revitalization",
    "Tedim sermon",
    "Zolai sermon",
    "Tedim devotional",
    "Zolai devotional",
    "Zomi worship songs",
    "Tedim Christian worship",
    "Zolai culture history",
    "Tedim oral history",
    "Zomi testimony",
    "Tedim Bible study",
    "Zolai blog",
    "Tedim article",
    "Zomi magazine",
    "Tedim church sermon",
]

MAX_RESULTS_PER_QUERY = 50  # URLs per search query (no limit)
REQUEST_TIMEOUT = 10.0
USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

# Search engine configurations
SEARCH_ENGINES = {
    'duckduckgo': {
        'url': 'https://html.duckduckgo.com/html/',
        'method': 'POST',
        'param': 'q',
        'result_selector': 'a.result__a',
        'href_attr': 'href',
    },
    'bing': {
        'url': 'https://www.bing.com/search',
        'method': 'GET',
        'param': 'q',
        'result_selector': 'li.b_algo h2 a',
        'href_attr': 'href',
    },
    'startpage': {
        'url': 'https://www.startpage.com/do/dsearch',
        'method': 'GET',
        'param': 'query',
        'result_selector': 'div.w-gl__result a.w-gl__result-title',
        'href_attr': 'href',
    },
    'mojeek': {
        'url': 'https://www.mojeek.com/search',
        'method': 'GET',
        'param': 'q',
        'result_selector': 'a.title',
        'href_attr': 'href',
    },
}

def search_duckduckgo(query, max_results=15):
    """Search DuckDuckGo HTML interface with simple paging."""
    urls = []
    seen = set()
    try:
        for offset in range(0, max_results, 30):
            resp = requests.post(
                'https://html.duckduckgo.com/html/',
                data={'q': query, 's': offset},
                headers={'User-Agent': USER_AGENT},
                timeout=REQUEST_TIMEOUT
            )
            soup = BeautifulSoup(resp.text, 'html.parser')
            page_found = 0
            for a in soup.select('a.result__a'):
                href = a.get('href', '')
                if '/l/?' in href:
                    from urllib.parse import unquote, parse_qs, urlparse
                    parsed = urlparse(href)
                    params = parse_qs(parsed.query)
                    if 'uddg' in params:
                        href = unquote(params['uddg'][0])
                if href.startswith('http') and href not in seen:
                    seen.add(href)
                    urls.append(href)
                    page_found += 1
                    if len(urls) >= max_results:
                        return urls
            if page_found == 0:
                break
            time.sleep(0.5)
    except Exception as e:
        print(f"  DuckDuckGo error: {e}")
    return urls


def search_bing(query, max_results=15):
    """Search Bing with paging."""
    urls = []
    seen = set()
    try:
        for first in range(1, max_results + 1, 10):
            resp = requests.get(
                'https://www.bing.com/search',
                params={'q': query, 'count': 10, 'first': first},
                headers={'User-Agent': USER_AGENT},
                timeout=REQUEST_TIMEOUT
            )
            soup = BeautifulSoup(resp.text, 'html.parser')
            page_found = 0
            for a in soup.select('li.b_algo h2 a'):
                href = a.get('href', '')
                if href.startswith('http') and href not in seen:
                    seen.add(href)
                    urls.append(href)
                    page_found += 1
                    if len(urls) >= max_results:
                        return urls
            if page_found == 0:
                break
            time.sleep(0.5)
    except Exception as e:
        print(f"  Bing error: {e}")
    return urls


def search_startpage(query, max_results=50):
    """Search Startpage (Google results, privacy-respecting)."""
    urls = []
    try:
        resp = requests.get(
            'https://www.startpage.com/do/dsearch',
            params={'query': query, 'num': max_results},
            headers={'User-Agent': USER_AGENT},
            timeout=REQUEST_TIMEOUT
        )
        soup = BeautifulSoup(resp.text, 'html.parser')
        for a in soup.select('a.w-gl__result-title')[:max_results]:
            href = a.get('href', '')
            if href.startswith('http'):
                urls.append(href)
    except Exception as e:
        print(f"  Startpage error: {e}")
    return urls


def search_mojeek(query, max_results=50):
    """Search Mojeek (independent crawler, no tracking)."""
    urls = []
    try:
        resp = requests.get(
            'https://www.mojeek.com/search',
            params={'q': query, 't': max_results},
            headers={'User-Agent': USER_AGENT},
            timeout=REQUEST_TIMEOUT
        )
        soup = BeautifulSoup(resp.text, 'html.parser')
        for a in soup.select('a.title')[:max_results]:
            href = a.get('href', '')
            if href.startswith('http'):
                urls.append(href)
    except Exception as e:
        print(f"  Mojeek error: {e}")
    return urls


def discover_links(queries=None, engines=None):
    """
    Search multiple engines for Zolai resources.
    Returns {url: {sources: [engines], query: str, score: float}}
    """
    if queries is None:
        queries = SEARCH_QUERIES
    if engines is None:
        engines = ['duckduckgo', 'bing']

    all_urls = defaultdict(lambda: {'sources': set(), 'queries': set()})
    
    searchers = {
        'duckduckgo': search_duckduckgo,
        'bing': search_bing,
    }

    total_found = 0
    for query in queries:
        print(f"\nSearching: '{query}'")
        for engine in engines:
            searcher = searchers.get(engine)
            if not searcher:
                continue
            
            try:
                urls = searcher(query, MAX_RESULTS_PER_QUERY)
                for url in urls:
                    # Clean URL
                    url = url.split('#')[0]  # Remove fragment
                    if url and url.startswith('http'):
                        all_urls[url]['sources'].add(engine)
                        all_urls[url]['queries'].add(query)
                print(f"  {engine}: {len(urls)} results")
                total_found += len(urls)
                time.sleep(1)  # Rate limit
            except Exception as e:
                print(f"  {engine} failed: {e}")
        
        time.sleep(1)  # Between queries

    # Score URLs by how many sources/queries found them
    scored_urls = []
    for url, info in all_urls.items():
        score = len(info['sources']) + len(info['queries']) * 0.5
        scored_urls.append({
            'url': url,
            'score': score,
            'sources': list(info['sources']),
            'queries': list(info['queries']),
        })

    # Sort by score (most confirmed first)
    scored_urls.sort(key=lambda x: x['score'], reverse=True)

    print(f"\n{'='*60}")
    print(f"DISCOVERY RESULTS")
    print(f"{'='*60}")
    print(f"Total unique URLs: {len(all_urls)}")
    print(f"Total search hits: {total_found}")
    print(f"\nTop 30 URLs (by confirmation score):")
    for i, entry in enumerate(scored_urls[:30], 1):
        domain = urlparse(entry['url']).netloc
        print(f"  {i:2d}. [{entry['score']:.1f}] {domain}")
        print(f"      {entry['url'][:80]}")
        print(f"      Sources: {', '.join(entry['sources'])} | Queries: {len(entry['queries'])}")

    return scored_urls


print("Link discovery functions defined.")
print(f"Search queries: {len(SEARCH_QUERIES)}")
print(f"Search engines: {', '.join(SEARCH_ENGINES.keys())}")

In [ ]:
#%% Cell 5: Run Link Discovery & Build Seed URL List

# Output dir (needed before Cell 7)
OUTPUT_DIR = WORK_DIR / "zolai_crawled_data"
OUTPUT_DIR.mkdir(exist_ok=True)

# Run discovery
discovered_urls = discover_links()

# Save discovered URLs
discovered_file = OUTPUT_DIR / "discovered_urls.json"
with open(discovered_file, 'w', encoding='utf-8') as f:
    json.dump(discovered_urls, f, ensure_ascii=False, indent=2)
print(f"\nSaved discovered URLs: {discovered_file}")

# Filter to best resources (score >= 2.0 means found by multiple sources)
CONFIRMED_THRESHOLD = 2.0
confirmed_urls = [u for u in discovered_urls if u['score'] >= CONFIRMED_THRESHOLD]

# Extract base URLs for crawling (homepages, not deep links)
seed_urls = []
seen_domains = set()
for entry in confirmed_urls:
    domain = urlparse(entry['url']).netloc
    if domain not in seen_domains:
        # Use homepage as seed
        seed_url = f"https://{domain}"
        seed_urls.append(seed_url)
        seen_domains.add(domain)

print(f"\n{'='*60}")
print(f"CONFIRMED RESOURCES FOR CRAWLING")
print(f"{'='*60}")
print(f"Confirmed URLs (score >= {CONFIRMED_THRESHOLD}): {len(confirmed_urls)}")
print(f"Unique domains (seeds): {len(seed_urls)}")
print(f"\nSeed domains:")
for i, url in enumerate(seed_urls, 1):
    domain = urlparse(url).netloc
    print(f"  {i:2d}. {domain}")

# Save seed URLs
seed_file = OUTPUT_DIR / "seed_urls.json"
with open(seed_file, 'w', encoding='utf-8') as f:
    json.dump(seed_urls, f, ensure_ascii=False, indent=2)
print(f"\nSaved seed URLs: {seed_file}")

In [ ]:
#%% Cell 6: Zolai Text Scoring System

# Rebuild lexicon if Cell 3 was not rerun after an error
if "zolai_vocab" not in globals():
    BIBLE_DATA = globals().get("BIBLE_DATA") or find_bible_data()
    if BIBLE_DATA is None:
        raise FileNotFoundError("Bible dataset not found for scorer bootstrap.")
    LEXICON = build_zolai_lexicon(BIBLE_DATA)
    patterns = LEXICON["patterns"]
    zolai_top_words = LEXICON["top_words"]
    zolai_top_bigrams = LEXICON["top_bigrams"]
    VERB_SUFFIXES = LEXICON["verb_suffixes"]
    zolai_verbs = LEXICON["verbs"]
    zolai_nouns = LEXICON["nouns"]
    zolai_adjectives = LEXICON["adjectives"]
    zolai_pronouns = LEXICON["pronouns"]
    zolai_particles = LEXICON["particles"]
    zolai_vocab = LEXICON["vocab"]

ENGLISH_STOPWORDS = {
    "the", "and", "of", "to", "that", "is", "in", "for", "with", "on",
    "this", "it", "you", "your", "be", "are", "was", "were", "from",
}


def ensure_lexicon_loaded():
    required = [
        "zolai_vocab", "zolai_top_bigrams", "zolai_verbs", "zolai_nouns",
        "zolai_particles", "zolai_adjectives", "VERB_SUFFIXES",
    ]
    missing = [name for name in required if name not in globals()]
    if not missing:
        return
    if "BIBLE_DATA" not in globals() or BIBLE_DATA is None:
        raise RuntimeError(f"Missing lexicon variables: {missing}. Run Cell 3 first.")
    rebuilt = build_zolai_lexicon(BIBLE_DATA)
    globals().update({
        "patterns": rebuilt["patterns"],
        "zolai_top_words": rebuilt["top_words"],
        "zolai_top_bigrams": rebuilt["top_bigrams"],
        "VERB_SUFFIXES": rebuilt["verb_suffixes"],
        "zolai_verbs": rebuilt["verbs"],
        "zolai_nouns": rebuilt["nouns"],
        "zolai_adjectives": rebuilt["adjectives"],
        "zolai_pronouns": rebuilt["pronouns"],
        "zolai_particles": rebuilt["particles"],
        "zolai_vocab": rebuilt["vocab"],
    })


def score_zolai_text(text, vocab=None, bigrams=None):
    """Score how likely text is Zolai using streamed Bible lexicon + grammar signals."""
    ensure_lexicon_loaded()
    if vocab is None:
        vocab = zolai_vocab
    if bigrams is None:
        bigrams = zolai_top_bigrams

    if not text or len(text) < 10:
        return {"score": 0.0, "is_zolai": False, "matches": 0, "total_words": 0}

    tokens = [normalize_zolai_token(tok) for tok in re.findall(r"[A-Za-z'\-]+", text)]
    tokens = [tok for tok in tokens if len(tok) >= 2 and tok != "note"]
    unique_words = set(tokens)
    total_words = len(unique_words)
    if total_words == 0:
        return {"score": 0.0, "is_zolai": False, "matches": 0, "total_words": 0}

    matched_words = unique_words & vocab
    verb_matches = unique_words & zolai_verbs
    noun_matches = unique_words & zolai_nouns
    particle_matches = unique_words & zolai_particles
    adjective_matches = unique_words & zolai_adjectives
    suffix_hits = sum(1 for tok in unique_words if any(tok.endswith(suf) for suf in VERB_SUFFIXES))

    word_score = len(matched_words) / total_words
    verb_score = min((len(verb_matches) + suffix_hits) / max(total_words, 1), 1.0)
    noun_score = len(noun_matches) / total_words
    particle_score = len(particle_matches) / total_words
    adjective_score = len(adjective_matches) / total_words

    text_lower = " ".join(tokens)
    bigram_matches = sum(1 for bg in bigrams if bg and bg in text_lower)
    bigram_score = min(bigram_matches / 8, 1.0)

    combined = (
        0.28 * word_score
        + 0.22 * verb_score
        + 0.15 * noun_score
        + 0.15 * particle_score
        + 0.08 * adjective_score
        + 0.12 * bigram_score
    )

    english_hits = len(unique_words & ENGLISH_STOPWORDS)
    if english_hits >= 4 and len(matched_words) <= 2:
        combined *= 0.35

    has_latin = bool(re.search(r'[a-zA-Z]', text))
    if not has_latin:
        combined *= 0.1

    return {
        "score": round(combined, 4),
        "is_zolai": combined >= 0.08,
        "matches": len(matched_words),
        "matched_words": sorted(list(matched_words))[:12],
        "verb_matches": sorted(list(verb_matches))[:12],
        "noun_matches": sorted(list(noun_matches))[:12],
        "particle_matches": sorted(list(particle_matches))[:12],
        "bigram_matches": bigram_matches,
        "suffix_hits": suffix_hits,
        "total_words": total_words,
    }


test_texts = [
    "A kipat cilin Pasian in vantung leh leitung a piangsak hi.",
    "In the beginning God created the heaven and the earth.",
    "This is random English text with no Zolai words at all.",
    "Pasian in khuavak hoih hi ci-in mu hi; Pasian in khuamial panin khuavak khenkhia hi.",
]

print("Zolai Scorer Test:")
for t in test_texts:
    result = score_zolai_text(t)
    print(f"  Score: {result['score']:.3f} | Zolai: {result['is_zolai']} | Matches: {result['matches']}")
    print(f"    Verbs: {', '.join(result['verb_matches']) or '-'}")
    print(f"    Nouns: {', '.join(result['noun_matches']) or '-'}")
    print(f"    Text: {t[:70]}...")


In [ ]:
#%% Cell 7: Web Crawler Configuration

# Seed URLs — Zolai/Tedim/Chin websites, sermons, culture, history
DEFAULT_SEED_URLS = [
    "https://zomidaily.com/",
    "https://www.tedimhymn.com/",
    "https://www.zomiworshipcollective.com/",
    "https://zomi.me/",
    "https://zogam.org/",
    "https://zomipedia.org/",
    "https://www.ebccdelhi.org/",
    "https://wol.jw.org/ctd/wol/d/r537/lp-ci/2018475",
    "https://wol.jw.org/ctd/wol/h/r537/lp-ci",
    "https://www.chinlanguages.org/",
    "https://zomielibrary.com/",
    "https://en.wikipedia.org/wiki/Tedim_language",
    "https://www.omniglot.com/writing/tedim.htm",
    "https://silverbaytranslations.wordpress.com/",
    "https://ethnomed.org/culture/chin/",
    "https://glosbe.com/en/ctd",
    "https://www.ethnologue.com/language/ctd/",
    "https://operation.asiaharvest.org/MYANMAR-Tedim-Chin.pdf",
    "https://polyglotclub.com/wiki/Language/Tedim-chin",
    "https://lingojam.com/zomi",
]
DISCOVERED_SEED_URLS = globals().get("seed_urls", [])
SEED_URLS = list(dict.fromkeys(DEFAULT_SEED_URLS + DISCOVERED_SEED_URLS))

# Crawler settings — aggressive for deep coverage
MAX_PAGES = 4000         # Max pages to crawl
MAX_DEPTH = 6            # Max link depth from seed (deeper)
REQUEST_DELAY = 0.5      # Seconds between requests (faster)
REQUEST_TIMEOUT = 20.0   # Request timeout
MIN_TEXT_LENGTH = 30     # Minimum text length to consider
SCORE_THRESHOLD = 0.05   # Lower threshold to capture more content
CHUNK_SIZE = 500         # Save every N lines

USER_AGENT = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"

# Output settings
OUTPUT_DIR = WORK_DIR / "zolai_crawled_data"
OUTPUT_DIR.mkdir(exist_ok=True)

# State management
STATE_FILE = WORK_DIR / "crawler_state.json"
LOG_FILE = WORK_DIR / "crawler.log"

def log(msg):
    ts = datetime.now().isoformat(timespec='seconds')
    line = f"[{ts}] {msg}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")

print(f"Output: {OUTPUT_DIR}")
print(f"Max pages: {MAX_PAGES}, Max depth: {MAX_DEPTH}")
print(f"Score threshold: {SCORE_THRESHOLD}")
print(f"Seed URLs: {len(SEED_URLS)}")
print(f"  Default seeds: {len(DEFAULT_SEED_URLS)}")
print(f"  Discovered seeds: {len(DISCOVERED_SEED_URLS)}")
for i, url in enumerate(SEED_URLS, 1):
    print(f"  {i:2d}. {url}")

In [ ]:
#%% Cell 8: Crawler Core

def extract_text_from_html(html, url):
    """Extract clean text from HTML, removing scripts, styles, nav elements."""
    soup = BeautifulSoup(html, 'html.parser')
    
    # Remove unwanted elements
    for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside', 'noscript']):
        tag.decompose()
    
    # Get text from main content areas
    content = soup.find(['main', 'article', 'div', 'body'])
    if content is None:
        content = soup
    
    text = content.get_text(separator=' ', strip=True)
    
    # Clean up whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


def extract_links(soup, base_url):
    """Extract internal links from a page."""
    links = []
    base_domain = urlparse(base_url).netloc
    
    for a_tag in soup.find_all('a', href=True):
        href = a_tag['href']
        full_url = urljoin(base_url, href)
        parsed = urlparse(full_url)
        
        # Only same-domain links
        if parsed.netloc == base_domain:
            # Remove fragments and normalize
            clean_url = f"{parsed.scheme}://{parsed.netloc}{parsed.path}"
            if clean_url.endswith('/'):
                clean_url = clean_url[:-1]
            links.append(clean_url)
    
    return list(set(links))


def fetch_page(url, session):
    """Fetch a page and return (html, status_code) or (None, error)."""
    try:
        resp = session.get(url, timeout=REQUEST_TIMEOUT, headers={
            'User-Agent': USER_AGENT,
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        })
        resp.raise_for_status()
        return resp.text, resp.status_code
    except Exception as e:
        return None, str(e)


def load_state():
    """Load crawler state for resume."""
    if STATE_FILE.exists():
        state = json.loads(STATE_FILE.read_text(encoding='utf-8'))
        log(f"Resumed from state: {len(state['visited'])} visited, {len(state['pending'])} pending")
        return state
    return {
        'visited': [],
        'pending': list(SEED_URLS),
        'depth_map': {url: 0 for url in SEED_URLS},
        'saved_count': 0,
        'chunk_idx': 0,
        'skipped_low_score': 0,
        'skipped_short': 0,
        'errors': 0,
    }


def save_state(state):
    """Save crawler state."""
    STATE_FILE.write_text(json.dumps(state, indent=2), encoding='utf-8')


print("Crawler core functions defined")

In [ ]:
#%% Cell 9: Run Crawler

state = load_state()
visited = set(state['visited'])
pending = state['pending']
depth_map = state.get('depth_map', {url: 0 for url in SEED_URLS})

saved_lines = []
session = requests.Session()
session.headers.update({'User-Agent': USER_AGENT})

log(f"Starting crawl: {len(pending)} pending, {len(visited)} already visited")

pages_crawled = 0
pages_saved = 0
chunk_idx = state.get('chunk_idx', 0)

while pending and pages_crawled < MAX_PAGES:
    url = pending.pop(0)
    
    if url in visited:
        continue
    
    current_depth = depth_map.get(url, 0)
    if current_depth > MAX_DEPTH:
        continue
    
    # Fetch page
    html, status = fetch_page(url, session)
    if html is None:
        state['errors'] += 1
        log(f"ERROR: {url} -> {status}")
        visited.add(url)
        continue
    
    # Extract text
    soup = BeautifulSoup(html, 'html.parser')
    text = extract_text_from_html(html, url)
    
    if len(text) < MIN_TEXT_LENGTH:
        state['skipped_short'] += 1
        visited.add(url)
        continue
    
    # Score text
    score_result = score_zolai_text(text)
    
    if score_result['is_zolai'] and score_result['score'] >= SCORE_THRESHOLD:
        # Split into sentences/paragraphs for finer granularity
        sentences = re.split(r'(?<=[.!?])\s+', text)
        for sent in sentences:
            sent = sent.strip()
            if len(sent) < 20:
                continue
            sent_score = score_zolai_text(sent)
            if sent_score['score'] >= SCORE_THRESHOLD:
                saved_lines.append({
                    'text': sent,
                    'source': 'web_crawl',
                    'url': url,
                    'score': sent_score['score'],
                    'matches': sent_score['matches'],
                    'matched_words': sent_score.get('matched_words', []),
                    'timestamp': datetime.now().isoformat(),
                })
        pages_saved += 1
        log(f"SAVED: {url} (score: {score_result['score']:.3f}, {len(saved_lines)} lines)")
    else:
        state['skipped_low_score'] += 1
    
    # Extract links
    new_links = extract_links(soup, url)
    for link in new_links:
        if link not in visited and link not in pending:
            pending.append(link)
            depth_map[link] = current_depth + 1
    
    visited.add(url)
    pages_crawled += 1
    
    # Save chunk periodically
    if len(saved_lines) >= CHUNK_SIZE:
        chunk_idx += 1
        chunk_file = OUTPUT_DIR / f"crawled_chunk_{chunk_idx:04d}.jsonl"
        with open(chunk_file, 'w', encoding='utf-8') as f:
            for line in saved_lines:
                f.write(json.dumps(line, ensure_ascii=False) + '\n')
        log(f"Saved chunk {chunk_idx}: {len(saved_lines)} lines -> {chunk_file.name}")
        saved_lines = []
    
    # Save state
    state['visited'] = list(visited)
    state['pending'] = pending
    state['depth_map'] = depth_map
    state['chunk_idx'] = chunk_idx
    save_state(state)
    
    # Rate limiting
    time.sleep(REQUEST_DELAY)

# Save remaining lines
if saved_lines:
    chunk_idx += 1
    chunk_file = OUTPUT_DIR / f"crawled_chunk_{chunk_idx:04d}.jsonl"
    with open(chunk_file, 'w', encoding='utf-8') as f:
        for line in saved_lines:
            f.write(json.dumps(line, ensure_ascii=False) + '\n')
    log(f"Saved final chunk {chunk_idx}: {len(saved_lines)} lines")

log(f"\nCrawl complete: {pages_crawled} pages, {pages_saved} saved, {len(visited)} visited")
log(f"Skipped: {state['skipped_low_score']} low score, {state['skipped_short']} short, {state['errors']} errors")

In [ ]:
#%% Cell 10: Combine Bible + Crawled Data

# Load Bible Zolai data
bible_zolai = []
bible_file = BIBLE_DATA / "bible_zolai_combined.jsonl"
if bible_file.exists():
    with open(bible_file, 'r', encoding='utf-8') as f:
        for line in f:
            entry = json.loads(line)
            entry['source'] = 'bible'
            bible_zolai.append(entry)
    log(f"Loaded {len(bible_zolai):,} Bible Zolai verses")

# Load crawled data
crawled_data = []
for chunk_file in sorted(OUTPUT_DIR.glob("crawled_chunk_*.jsonl")):
    with open(chunk_file, 'r', encoding='utf-8') as f:
        for line in f:
            crawled_data.append(json.loads(line))
log(f"Loaded {len(crawled_data):,} crawled lines from {len(list(OUTPUT_DIR.glob('crawled_chunk_*.jsonl')))} chunks")

# Combine and deduplicate
combined = bible_zolai + crawled_data
seen_texts = set()
deduped = []
dupes = 0
for entry in combined:
    text_key = entry['text'].strip().lower()
    h = hashlib.md5(text_key.encode('utf-8')).hexdigest()
    if h not in seen_texts:
        seen_texts.add(h)
        deduped.append(entry)
    else:
        dupes += 1

log(f"Combined: {len(combined):,} -> {len(deduped):,} unique ({dupes:,} duplicates removed)")

# Save combined dataset
combined_file = OUTPUT_DIR / "zolai_combined_training.jsonl"
with open(combined_file, 'w', encoding='utf-8') as f:
    for entry in deduped:
        f.write(json.dumps(entry, ensure_ascii=False) + '\n')

# Stats
bible_count = sum(1 for e in deduped if e.get('source') == 'bible')
web_count = sum(1 for e in deduped if e.get('source') == 'web_crawl')

log(f"\nCombined dataset:")
log(f"  Bible verses: {bible_count:,}")
log(f"  Web lines:    {web_count:,}")
log(f"  Total:        {len(deduped):,}")
log(f"  Saved to: {combined_file}")

In [ ]:
#%% Cell 11: Create HuggingFace Dataset
try:
    from datasets import Dataset, DatasetDict

    # Create training dataset
    train_rows = []
    for entry in deduped:
        train_rows.append({
            'text': entry['text'],
            'source': entry.get('source', 'unknown'),
            'language': 'zolai',
            'dialect': 'tedim',
        })

    ds = Dataset.from_list(train_rows)
    ds.save_to_disk(str(OUTPUT_DIR / "hf_zolai_training"))
    print(f"Saved HuggingFace dataset: {OUTPUT_DIR / 'hf_zolai_training'}")
    print(f"  Rows: {len(ds):,}")
    print(f"  Features: {ds.features}")
except ImportError:
    print("datasets library not available, skipping HF export")

In [ ]:
#%% Cell 12: ZIP & Upload
import zipfile

ZIP_NAME = "zolai_crawled_dataset.zip"
ZIP_PATH = WORK_DIR / ZIP_NAME

print(f"Creating ZIP: {ZIP_PATH}")
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in sorted(OUTPUT_DIR.rglob("*")):
        if fpath.is_file():
            arcname = str(fpath.relative_to(OUTPUT_DIR))
            zf.write(fpath, arcname)

zip_size = ZIP_PATH.stat().st_size
print(f"\nZIP created: {ZIP_NAME} ({zip_size/1024/1024:.1f} MB)")

if IS_KAGGLE:
    print("\n" + "=" * 60)
    print("KAGGLE DATASET UPLOAD INSTRUCTIONS")
    print("=" * 60)
    print("""
Option A — Download ZIP + Upload as New Dataset:
  1. Download zolai_crawled_dataset.zip from Output panel
  2. Go to kaggle.com → Datasets → New Dataset
  3. Upload the ZIP, name it 'zolai-crawled-dataset'
  4. Set License, make it Public/Private

Option B — Use Kaggle API in this notebook:
  1. Add your kaggle.json as a Kaggle Secret
  2. Set UPLOAD_TO_KAGGLE = True below

Option C — Save as Notebook Output Dataset:
  1. Save Version of this notebook
  2. The /kaggle/working/ output becomes a dataset
  3. Other notebooks can add it via Add Input
""")

    UPLOAD_TO_KAGGLE = False
    if UPLOAD_TO_KAGGLE:
        !pip install -q kaggle 2>/dev/null
        import kaggle
        dataset_slug = "peterpausianlian/zolai-crawled-dataset"
        print(f"Uploading to {dataset_slug}...")
        kaggle.api.dataset_create_version(
            dataset_slug,
            folder=str(OUTPUT_DIR),
            version_notes="Auto-generated by zolai-web-crawler-v3"
        )
        print("Upload complete!")

# Next Steps & Reuse Guide

## Download Your Dataset
1. **ZIP file**: `zolai_crawled_dataset.zip` is in `/kaggle/working/` — download from Output panel
2. **Save Version**: Click **Save Version** → output becomes a Kaggle dataset automatically
3. **Upload as Dataset**: Go to kaggle.com → Datasets → New → upload the ZIP

## Reuse in Other Notebooks
```python
# Add the dataset via Add Input, then:
CRAWLED_DATA = Path("/kaggle/input/YOUR_USERNAME/zolai-crawled-dataset")

# Load combined training data
import json
with open(CRAWLED_DATA / "zolai_combined_training.jsonl") as f:
    lines = [json.loads(line) for line in f]

# Load HuggingFace dataset
from datasets import load_from_disk
ds = load_from_disk(str(CRAWLED_DATA / "hf_zolai_training"))
print(f"Total Zolai texts: {len(ds):,}")
```

## Which Files for What

| File | Use Case |
|------|----------|
| `zolai_combined_training.jsonl` | **Primary training corpus** — Bible + Web combined |
| `crawled_chunk_*.jsonl` | Raw crawled data chunks (with scores and URLs) |
| `hf_zolai_training/` | **HuggingFace format** — `load_from_disk()` ready |
| `crawler_state.json` | Resume state — use to continue crawling |
| `language_patterns.json` | Bible vocabulary reference (copied from Bible dataset) |

## Next Steps

### 1. Fine-Tune Qwen on Combined Data
```python
from datasets import load_from_disk
ds = load_from_disk(str(CRAWLED_DATA / "hf_zolai_training"))
# Use with TRL SFTTrainer
```

### 2. Expand Crawling
- Add more seed URLs to `SEED_URLS` in Cell 4
- Increase `MAX_PAGES` and `MAX_DEPTH`
- Run again — state is saved, will resume from where it left off

### 3. Improve Scoring
- Update `language_patterns.json` with more vocabulary
- Adjust `SCORE_THRESHOLD` based on results
- Add custom word lists for specific domains